# AgentTorch Media Image Debug

这个 notebook 用来排查和演示 `generate_image` 链路，覆盖四件事：

1. 加载并检查 `.env` 图片配置
2. 直接调用 `OpenAIModel.generate_image(...)`
3. 直接执行 `generate_image` 工具，隔离 tool 层
4. 用 `agent.run(..., stream=True)` 查看真实工具调用进度与结果

建议先跑前 3 步，确认图片接口本身通了，再看 agent 工具调用行为。


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from uuid import uuid4

from IPython.display import Image as IPyImage, display

from agentorch import OpenAIModel, create_agent
from agentorch.config.settings import initialize_environment
from agentorch.tools import ToolRegistry, create_generate_image_tool

ROOT = Path.cwd()
ENV_PATH = ROOT / '.env'
ARTIFACT_DIR = ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

initialize_environment(ENV_PATH, overwrite=True)

print('ROOT ->', ROOT)
print('ENV_PATH ->', ENV_PATH, 'exists=', ENV_PATH.exists())


In [ ]:
def mask_secret(value: str | None) -> str:
    if not value:
        return '<missing>'
    value = value.strip()
    if len(value) <= 8:
        return '*' * len(value)
    return value[:4] + '...' + value[-4:]


def env_snapshot() -> dict[str, str]:
    keys = [
        'APIYI_KEY',
        'APIYI_KEY_image',
        'APIYI_IMAGE_API_KEY',
        'APIYI_GEN_BASE_URL',
        'APIYI_IMAGE_BASEURL',
        'APIYI_IMAGE_BASE_URL',
        'APIYI_GEN_MODEL',
        'APIYI_IMAGE_MODEL',
        'OPENAI_IMAGE_API_KEY',
        'OPENAI_IMAGE_BASE_URL',
        'OPENAI_IMAGE_MODEL',
    ]
    snapshot = {}
    for key in keys:
        raw = os.getenv(key)
        snapshot[key] = mask_secret(raw) if 'KEY' in key else (raw or '<missing>')
    return snapshot


print(json.dumps(env_snapshot(), ensure_ascii=False, indent=2))
print('说明: 当前代码已经兼容 APIYI_IMAGE_BASEURL / APIYI_IMAGE_MODEL 这类旧键名。')
print('如果图片请求仍失败，请优先排查 token 是否真能调用图片接口。')


In [ ]:
def build_image_model() -> OpenAIModel:
    model = OpenAIModel(model='gpt-4.1-mini', image_timeout=20.0)
    resolved = {
        'image_api_key': mask_secret(model.config.image_api_key or model.config.api_key),
        'image_base_url': model.config.image_base_url,
        'image_model': model.config.image_model,
        'image_fallback_models': list(model.config.image_fallback_models),
        'image_timeout': model.config.image_timeout,
    }
    print(json.dumps(resolved, ensure_ascii=False, indent=2))
    return model


preview_model = build_image_model()
await preview_model.aclose()


In [ ]:
async def generate_image_direct(prompt: str, output_path: str = 'artifacts/direct_yangguo_diaoxiong.png'):
    model = build_image_model()
    try:
        result = await model.generate_image(prompt, output_path=ROOT / output_path)
        print('[direct] success ->', result.model_dump())
        display(IPyImage(filename=result.output_path))
        return result
    except Exception as exc:  # noqa: BLE001
        print('[direct] failed ->', type(exc).__name__)
        print(str(exc))
        raise
    finally:
        await model.aclose()


DIRECT_PROMPT = '杨过与雕兄并肩作战，襄阳大战，写实风格，电影感，史诗战场，超清细节。'
# 取消下一行注释，直接测试图片生成接口
# direct_result = await generate_image_direct(DIRECT_PROMPT)


In [ ]:
async def generate_image_via_tool(prompt: str, output_path: str = 'artifacts/tool_yangguo_diaoxiong.png'):
    model = build_image_model()
    try:
        registry = ToolRegistry.from_tools(create_generate_image_tool(model, ROOT))
        result = await registry.execute('generate_image', {'prompt': prompt, 'output_path': output_path})
        print('[tool] success ->', result.success)
        print(json.dumps(result.data, ensure_ascii=False, indent=2))
        if result.success and result.data.get('output_path'):
            display(IPyImage(filename=result.data['output_path']))
        return result
    finally:
        await model.aclose()


# 取消下一行注释，直接测试 tool 层
# tool_result = await generate_image_via_tool(DIRECT_PROMPT)


In [ ]:
media_agent = create_agent(
    model=OpenAIModel(model='gpt-4.1-mini', image_timeout=20.0),
    tool_bundles={
        'include_filesystem': False,
        'include_execution': False,
        'include_git': False,
        'include_media': True,
    },
    workspace_root=ROOT,
    system_prompt=(
        '你是一个多媒体工具 agent。遇到生成图片请求时，必须直接调用一次 generate_image 工具，'
        '并显式提供 prompt 和 output_path 参数；完成后返回实际 output_path。'
    ),
)

print('agent media tools ->', media_agent.export_blueprint()['runtime']['tools'])


In [ ]:
async def run_media_agent_with_progress(
    *,
    scene_prompt: str,
    output_path: str = 'artifacts/agent_yangguo_diaoxiong.png',
):
    thread_id = f'nb-media-agent-{uuid4().hex[:8]}'
    final_result = None
    tool_payloads = []

    user_prompt = (
        '请直接调用一次 generate_image 工具。'
        f'prompt 使用: {scene_prompt} '
        f'output_path 使用: {output_path} '
        '不要先追问，不要省略工具参数。'
    )

    async for event in media_agent.run(user_prompt, thread_id=thread_id, stream=True):
        if event.event_type == 'run_started':
            print(f'[run_started] thread_id={event.thread_id}')

        elif event.event_type == 'tool_called':
            print('[tool_called]')
            print('payload ->', json.dumps(event.payload, ensure_ascii=False, indent=2))
            for call in event.tool_calls:
                print('tool_name ->', call.name)
                print('arguments ->', call.arguments)

        elif event.event_type == 'tool_result':
            print('[tool_result]')
            print(json.dumps(event.payload, ensure_ascii=False, indent=2))
            tool_payloads.append(event.payload)
            if not event.payload.get('is_error'):
                output = event.payload.get('output') or {}
                actual_path = output.get('output_path')
                if actual_path and Path(actual_path).exists():
                    print('[display] ->', actual_path)
                    display(IPyImage(filename=actual_path))

        elif event.event_type == 'final_result' and event.result is not None:
            final_result = event.result
            print('[final_result]')
            print(event.result.output_text)

    return {'thread_id': thread_id, 'final_result': final_result, 'tool_payloads': tool_payloads}


# 取消下一行注释，查看 agent 的真实工具调用过程
# agent_debug = await run_media_agent_with_progress(scene_prompt=DIRECT_PROMPT)


In [ ]:
def list_recent_generated_images(limit: int = 20):
    candidates = []
    for pattern in ('artifacts/*.png', 'artifacts/*.jpg', 'artifacts/*.jpeg', 'artifacts/*.webp', '.agentorch/images/*'):
        candidates.extend(ROOT.glob(pattern))
    candidates = sorted({path.resolve() for path in candidates if path.is_file()}, key=lambda p: p.stat().st_mtime, reverse=True)
    for path in candidates[:limit]:
        print(path)
    return candidates[:limit]


list_recent_generated_images()


## 建议的排查顺序

1. 先运行 `build_image_model()`，确认 `image_base_url` / `image_model` / key 已正确解析。
2. 再运行 `generate_image_direct(...)`。如果这里都失败，优先排查 token 和图片接口权限。
3. 如果直连成功，再运行 `generate_image_via_tool(...)`，确认 tool 层正常。
4. 最后才跑 `run_media_agent_with_progress(...)`，查看 agent 是否真的把 `prompt` / `output_path` 传给了工具。

如果 `tool_result` 里出现 `is_error=true`，不要只看最后自然语言回复，直接看那个错误字段。
